# 16 - Git workflows, CI/CD and Docker

**Sources:** Bootcamp **BP8** (git workflows) and **BP9** (CI/CD: GitLab CI, Docker).

The git part runs real `git` commands in a throw-away temporary repository, so you can see the output
without touching this repository. Docker and CI files are shown as text (they need a server / the Docker daemon).

In [1]:
import subprocess
import tempfile
import textwrap
from pathlib import Path

work = Path(tempfile.mkdtemp(prefix="git_demo_"))


def git(*args, check=True):
    """Run `git <args>` inside the demo repo and print what it says."""
    result = subprocess.run(
        ["git", "-c", "user.name=Demo", "-c", "user.email=demo@example.com", "-c", "core.autocrlf=false", *args],
        cwd=work, capture_output=True, text=True,
    )
    text = (result.stdout + result.stderr).strip()
    print(f"$ git {' '.join(args)}")
    if text:
        print(textwrap.indent(text, "  "))
    if check and result.returncode != 0:
        print(f"  (exit code {result.returncode})")


def write(name, content):
    (work / name).write_text(content, encoding="utf-8", newline="\n")

## 1. The three areas and the basic cycle
*working tree* -> `git add` -> *staging area (index)* -> `git commit` -> *repository history*.

In [2]:
git("init", "-b", "main")
write("app.py", "def greet(name):\n    return f'Hello {name}'\n")
git("status", "--short")
git("add", "app.py")
git("status", "--short")
git("commit", "-m", "Add greet function")
git("log", "--oneline")

$ git init -b main
  Initialized empty Git repository in C:/Users/leire/AppData/Local/Temp/git_demo_inwa53mc/.git/
$ git status --short
  ?? app.py
$ git add app.py
$ git status --short
  A  app.py
$ git commit -m Add greet function
  [main (root-commit) 82e6ad6] Add greet function
   1 file changed, 2 insertions(+)
   create mode 100644 app.py


$ git log --oneline
  82e6ad6 Add greet function


## 2. Feature-branch workflow
`main` is always releasable. Every change lives on a short-lived `feature/<topic>` branch and returns through a
pull / merge request (review + green pipeline).

In [3]:
git("switch", "-c", "feature/farewell")
write("app.py", "def greet(name):\n    return f'Hello {name}'\n\n\ndef farewell(name):\n    return f'Bye {name}'\n")
git("commit", "-am", "Add farewell function")
git("switch", "main")
write("README.md", "# Demo\n")
git("add", "README.md")
git("commit", "-m", "Add README")
# main and the feature branch have diverged: a merge commit joins them (--no-ff always creates one)
git("merge", "--no-ff", "feature/farewell", "-m", "Merge feature/farewell")
git("log", "--oneline", "--graph", "--all")

$ git switch -c feature/farewell
  Switched to a new branch 'feature/farewell'
$ git commit -am Add farewell function
  [feature/farewell ed9459e] Add farewell function
   1 file changed, 4 insertions(+)
$ git switch main
  Switched to branch 'main'


$ git add README.md


$ git commit -m Add README
  [main 1ed6499] Add README
   1 file changed, 1 insertion(+)
   create mode 100644 README.md
$ git merge --no-ff feature/farewell -m Merge feature/farewell
  Merge made by the 'ort' strategy.
   app.py | 4 ++++
   1 file changed, 4 insertions(+)
$ git log --oneline --graph --all
  *   a682019 Merge feature/farewell
  |\  
  | * ed9459e Add farewell function
  * | 1ed6499 Add README
  |/  
  * 82e6ad6 Add greet function


## 3. Merge vs rebase
* `merge` keeps the history exactly as it happened (a merge commit appears).
* `rebase` replays *your* commits on top of the target branch -> linear history. It rewrites commits, so
  **never rebase branches that other people already use**.

In [4]:
git("switch", "-c", "feature/shout")
write("shout.py", "def shout(text):\n    return text.upper() + '!'\n")
git("add", "shout.py")
git("commit", "-m", "Add shout")
git("switch", "main")
write("README.md", "# Demo\n\nSmall demo project.\n")
git("commit", "-am", "Extend README")
git("switch", "feature/shout")
git("rebase", "main")
git("log", "--oneline", "--graph", "--all")
git("switch", "main")
git("merge", "--ff-only", "feature/shout")      # fast-forward: no merge commit, history stays linear
git("log", "--oneline")

$ git switch -c feature/shout
  Switched to a new branch 'feature/shout'


$ git add shout.py
$ git commit -m Add shout
  [feature/shout c0f1ab4] Add shout
   1 file changed, 2 insertions(+)
   create mode 100644 shout.py
$ git switch main
  Switched to branch 'main'


$ git commit -am Extend README
  [main 86d3fb1] Extend README
   1 file changed, 2 insertions(+)
$ git switch feature/shout
  Switched to branch 'feature/shout'
$ git rebase main
  Rebasing (1/1)

  Successfully rebased and updated refs/heads/feature/shout.
$ git log --oneline --graph --all
  * 7ee2cfc Add shout
  * 86d3fb1 Extend README
  *   a682019 Merge feature/farewell
  |\  
  | * ed9459e Add farewell function
  * | 1ed6499 Add README
  |/  
  * 82e6ad6 Add greet function


$ git switch main
  Switched to branch 'main'
$ git merge --ff-only feature/shout
  Updating 86d3fb1..7ee2cfc
  Fast-forward
   shout.py | 2 ++
   1 file changed, 2 insertions(+)
   create mode 100644 shout.py
$ git log --oneline
  7ee2cfc Add shout
  86d3fb1 Extend README
  a682019 Merge feature/farewell
  1ed6499 Add README
  ed9459e Add farewell function
  82e6ad6 Add greet function


## 4. Conflicts
Two branches change the same line. Git stops and marks the file with `<<<<<<<`, `=======`, `>>>>>>>`.
Fix: edit the file, `git add`, then `git commit` (merge) or `git rebase --continue`.

In [5]:
git("switch", "-c", "feature/polite")
write("app.py", (work / "app.py").read_text().replace("Hello", "Good morning"))
git("commit", "-am", "Polite greeting")
git("switch", "main")
write("app.py", (work / "app.py").read_text().replace("Hello", "Hi"))
git("commit", "-am", "Casual greeting")
git("merge", "feature/polite", check=False)
print((work / "app.py").read_text())

$ git switch -c feature/polite
  Switched to a new branch 'feature/polite'


$ git commit -am Polite greeting
  [feature/polite 4f88b0b] Polite greeting
   1 file changed, 1 insertion(+), 1 deletion(-)
$ git switch main
  Switched to branch 'main'
$ git commit -am Casual greeting
  [main 0a26719] Casual greeting
   1 file changed, 1 insertion(+), 1 deletion(-)


$ git merge feature/polite
  Auto-merging app.py
  CONFLICT (content): Merge conflict in app.py
  Automatic merge failed; fix conflicts and then commit the result.


def greet(name):
<<<<<<< HEAD
    return f'Hi {name}'
    return f'Good morning {name}'
>>>>>>> feature/polite


def farewell(name):
    return f'Bye {name}'



In [6]:
# resolve: keep the polite version, then finish the merge
write("app.py", "def greet(name):\n    return f'Good morning {name}'\n\n\ndef farewell(name):\n    return f'Bye {name}'\n")
git("add", "app.py")
git("commit", "-m", "Merge feature/polite (keep polite greeting)")
git("log", "--oneline", "--graph", "-5")

$ git add app.py
$ git commit -m Merge feature/polite (keep polite greeting)
  [main 6dd6dc5] Merge feature/polite (keep polite greeting)
$ git log --oneline --graph -5
  *   6dd6dc5 Merge feature/polite (keep polite greeting)
  |\  
  | * 4f88b0b Polite greeting
  * | 0a26719 Casual greeting
  |/  
  * 7ee2cfc Add shout
  * 86d3fb1 Extend README


## 5. Handy tools
* `git stash` / `git stash pop` - park unfinished work.
* `git restore <file>` - discard changes of a file; `git restore --staged <file>` - unstage.
* `git cherry-pick <sha>` - copy a single commit to the current branch.
* `git revert <sha>` - new commit that undoes an old one (safe for shared history).
* `git reset --hard <sha>` - move the branch (and discard work!); `git reflog` records every move so you can undo it.
* `git bisect` - binary search for the commit that introduced a bug.

In [7]:
write("app.py", "def greet(name):\n    return 'unfinished'\n")
git("stash")
git("status", "--short")
git("stash", "pop")
git("restore", "app.py")
write("notes.txt", "a mistake\n")
git("add", "notes.txt")
git("commit", "-m", "Add notes by mistake")
git("revert", "--no-edit", "HEAD")             # a *new* commit that undoes the previous one
git("log", "--oneline", "-3")
git("reflog", "-3")

$ git stash
  Saved working directory and index state WIP on main: 6dd6dc5 Merge feature/polite (keep polite greeting)
$ git status --short
$ git stash pop
  On branch main
  Changes not staged for commit:
    (use "git add <file>..." to update what will be committed)
    (use "git restore <file>..." to discard changes in working directory)
  	modified:   app.py

  no changes added to commit (use "git add" and/or "git commit -a")
  Dropped refs/stash@{0} (11083fae1889cda78119422b0a19d2551130427d)
$ git restore app.py
$ git add notes.txt


$ git commit -m Add notes by mistake
  [main ceacbbe] Add notes by mistake
   1 file changed, 1 insertion(+)
   create mode 100644 notes.txt
$ git revert --no-edit HEAD
  [main 41875ca] Revert "Add notes by mistake"
   Date: Mon Oct 12 20:34:37 2026 +0200
   1 file changed, 1 deletion(-)
   delete mode 100644 notes.txt
$ git log --oneline -3
  41875ca Revert "Add notes by mistake"
  ceacbbe Add notes by mistake
  6dd6dc5 Merge feature/polite (keep polite greeting)
$ git reflog -3
  41875ca HEAD@{0}: revert: Revert "Add notes by mistake"
  ceacbbe HEAD@{1}: commit: Add notes by mistake
  6dd6dc5 HEAD@{2}: reset: moving to HEAD


## 6. Other branching strategies (BP8)
* **Gitflow**: long-lived `develop` + `release/*` + `hotfix/*` branches. Heavy, good for versioned releases.
* **GitHub flow** (the one above): `main` + short feature branches + pull requests.
* **Trunk-based development**: tiny commits straight to `main` behind feature flags; needs strong CI.

Good commit habits: one logical change per commit, imperative subject line under ~50 characters
("Add farewell function"), body explains *why*. A `.gitignore` keeps `.venv/`, `__pycache__/`, secrets and
data out of the history (never commit passwords or tokens: rotate them if you did).

## 7. CI/CD
**Continuous Integration**: every push automatically builds, lints and tests the code, so broken changes are found
within minutes. **Continuous Delivery / Deployment**: green builds are packaged (e.g. as a Docker image) and
released automatically or with one click.

A typical **GitLab** pipeline (`.gitlab-ci.yml`): stages run in order, jobs inside a stage run in parallel.

In [8]:
gitlab_ci = """
image: python:3.12-slim

stages: [quality, test, build]

before_script:
  - python -m pip install -r requirements.txt

format:
  stage: quality
  script: [black --check ., isort --check .]

static:
  stage: quality
  script: [flake8 src, mypy src, bandit -r src -q]

unit-tests:
  stage: test
  script: [pytest --cov=src --cov-report=term --cov-fail-under=80]
  coverage: '/TOTAL.*\\s+(\\d+%)$/'

docker-image:
  stage: build
  image: docker:24
  services: [docker:dind]
  script: [docker build -t $CI_REGISTRY_IMAGE:$CI_COMMIT_SHORT_SHA .]
  only: [main]
"""
print(gitlab_ci)


image: python:3.12-slim

stages: [quality, test, build]

before_script:
  - python -m pip install -r requirements.txt

format:
  stage: quality
  script: [black --check ., isort --check .]

static:
  stage: quality
  script: [flake8 src, mypy src, bandit -r src -q]

unit-tests:
  stage: test
  script: [pytest --cov=src --cov-report=term --cov-fail-under=80]
  coverage: '/TOTAL.*\s+(\d+%)$/'

docker-image:
  stage: build
  image: docker:24
  services: [docker:dind]
  script: [docker build -t $CI_REGISTRY_IMAGE:$CI_COMMIT_SHORT_SHA .]
  only: [main]



The same idea with **GitHub Actions** (`.github/workflows/ci.yml`):

In [9]:
github_actions = """
name: CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip
      - run: python -m pip install -r requirements.txt
      - run: black --check .
      - run: flake8
      - run: pytest -q
"""
print(github_actions)


name: CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip
      - run: python -m pip install -r requirements.txt
      - run: black --check .
      - run: flake8
      - run: pytest -q



CI tips: keep the pipeline fast (cache dependencies, run cheap checks first), make it **reproducible** (pin versions in
`requirements.txt`), never put secrets in the YAML (use the platform's *secret variables*), and make
"pipeline is green" a requirement for merging (branch protection).

## 8. Docker in one minute
An **image** is an immutable, layered filesystem snapshot built from a recipe (`Dockerfile`); a **container** is a
running instance of an image. Containers give every developer and the CI server the same environment.

In [10]:
dockerfile = """
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt   # own layer: cached until requirements.txt changes
COPY src/ src/
CMD ["python", "-m", "src.main"]
"""
print(dockerfile)


FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt   # own layer: cached until requirements.txt changes
COPY src/ src/
CMD ["python", "-m", "src.main"]



```powershell
docker build -t myapp .                 # build the image
docker run --rm -it myapp               # run it, delete the container afterwards
docker run --rm -v ${PWD}:/app myapp    # mount the current folder (live code editing)
docker compose up                       # several containers (app + database) from docker-compose.yml
```
A **dev container** (VS Code / Cursor: *Dev Containers* extension, `.devcontainer/devcontainer.json`) opens your
project inside a container, so the whole team shares one toolchain - the approach used in the bootcamp.
Also add a `.dockerignore` (`.venv`, `.git`, `__pycache__`) to keep the build context small.

## Exercises
**A.** In a scratch folder: create a repo, make three commits on `main`, branch, change the same line on both branches,
produce a conflict and resolve it. Then undo the merge with `git reflog` + `git reset --hard`.
**B.** Turn the commits of a feature branch into a single commit with `git rebase -i main` (squash).
**C.** Add a `.github/workflows/ci.yml` to this repository that runs `python -m pytest -q`
(the practice tests report not-implemented stubs as *skipped*, so the pipeline is green from the start).
**D.** Write a `Dockerfile` for `projects/01_expense_tracker` that runs its tests, and a `.dockerignore`.
**E.** Protect `main` on your Git host: require a pull request and a passing pipeline before merging.

In [11]:
# Your turn

In [12]:
import shutil
import stat


def _force_remove(func, path, _exc):
    Path(path).chmod(stat.S_IWRITE)      # git object files are read-only on Windows
    func(path)


shutil.rmtree(work, onexc=_force_remove)
print("demo repository removed")

demo repository removed
